# Nivo Project

In [ ]:
# ================================================================
# CONFIG: set DATA_DIR to the local path where the IMC data lives.
# Expected contents: outs/phenotyped-obs.csv, outs/combat_all_markers.csv,
# CN_data/pheno_8clus-cells_data.csv, CN_data/pheno_8clus-sample_pct.csv,
# nivo_md.csv, nivo_clinical.csv, nivo_MSPike.csv, and the raw IMC images.
# Processed CSVs are deposited at the Zenodo DOI listed in the README.
# ================================================================
import os
DATA_DIR = os.environ.get('NIVO_IMC_DATA_DIR', DATA_DIR + '')
if not DATA_DIR.endswith('/'):
    DATA_DIR += '/'
dir_ = DATA_DIR  # alias used throughout the notebook
print(f'Using DATA_DIR = {DATA_DIR}')

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import os
import tifffile

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## Funs

In [ ]:
from collections import defaultdict
from scipy import ndimage
from scipy.sparse import csr_matrix,coo_matrix
from scipy.spatial import cKDTree
from skimage.segmentation import find_boundaries
import networkx as nx #docker:cellpose_gpu

# Percentile-normalise expression
def pctnorm(img, pct_lo = 1, pct_hi = 99.7):
    from skimage import exposure
    p_low, p_high = np.percentile(img, [pct_lo, pct_hi])
    return exposure.rescale_intensity(img, in_range=(p_low, p_high), out_range=(0, 1))

# Cell metadata
def summarise_cellmd(mask):
    boundaries = find_boundaries(mask)
    cell_ids = np.unique(mask)[1:]
    sizes = ndimage.sum(np.ones_like(mask), mask, cell_ids)
    perims = ndimage.sum(boundaries, mask, cell_ids)
    centroids = ndimage.center_of_mass(np.ones_like(mask), mask, cell_ids)
    y_coords = [centroid[0] for centroid in centroids]
    x_coords = [centroid[1] for centroid in centroids]
    return pd.DataFrame({'size': sizes, 'perim': perims, 'x':x_coords, 'y':y_coords}, index=cell_ids)

# Summarise expression
def summarise_expression(mask, expr):
    # get non-background mask positions
    valid_mask = mask > 0
    mask_flat = mask[valid_mask]
    # unique cell_id
    cell_ids = np.unique(mask_flat)
    # flatten expression data for cell_id positions
    expr_flat = expr[:, valid_mask]  #(n_channels, n_valid_pixels)
    # Use pandas groupby for efficient aggregation
    df = pd.DataFrame(expr_flat.T, index=mask_flat)
    grouped_means = df.groupby(level=0).mean() 
    return cell_ids, grouped_means.values


def img_map(img,figsize=(3,3),cmap='viridis',title=None, origin='lower',labels=False,cbar=False,save=None):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(img, cmap=cmap, origin=origin)
    ax.set_title(title)
    if not labels:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    if cbar:
        plt.colorbar(im, ax=ax)
    if save is not None:
        plt.savefig(save, dpi=300, bbox_inches='tight', transparent=False) 
    plt.show()



In [ ]:
# Border graph functions

import os
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.sparse import csr_matrix,coo_matrix
from scipy.spatial import cKDTree
from skimage.segmentation import find_boundaries

# Calculate cell border lengths
def calculate_cell_border_lengths(mask):
    
    cell_ids = np.unique(mask)
    if 0 in cell_ids:
        cell_ids = cell_ids[cell_ids != 0]  # Remove background
    # sort IDs to ensure consistent ordering
    cell_ids = np.sort(cell_ids)
    n_cells = len(cell_ids)
    # mapping from cell ID to matrix index
    cell_to_index = {cell_id: idx for idx, cell_id in enumerate(cell_ids)}
    # dictionary to store border lengths between cell pairs (using matrix indices)
    border_lengths = defaultdict(float) 
    # horizontal boundaries (between horizontally adjacent pixels)
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1] - 1):
            cell1 = mask[i, j]
            cell2 = mask[i, j + 1]
            # only count if both cells non-background and different
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                # convert cell IDs to matrix indices
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                # use tuple with smaller index first for consistency
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0  # Each pixel boundary has length 1
    # vertical boundaries (between vertically adjacent pixels)
    for i in range(mask.shape[0] - 1):
        for j in range(mask.shape[1]):
            cell1 = mask[i, j]
            cell2 = mask[i + 1, j] 
            # only count if both cells are non-background and different
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                # convert cell IDs to matrix indices
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                # use tuple with smaller index first for consistency
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0  # Each pixel boundary has length 1
    # convert to sparse matrix format
    if not border_lengths:
        # No borders found, return empty sparse matrix
        return coo_matrix((n_cells, n_cells))
    # create coordinate arrays for sparse matrix
    rows = []
    cols = []
    data = []
    for (idx1, idx2), length in border_lengths.items():
        # Add both (i,j) and (j,i) entries to make the matrix symmetric
        rows.extend([idx1, idx2])
        cols.extend([idx2, idx1])
        data.extend([length, length])
    # Create sparse matrix with dimensions N x N where N is number of cells
    border_matrix = coo_matrix((data, (rows, cols)), shape=(n_cells, n_cells))
    return border_matrix,cell_ids

def calculate_cell_border_lengths_dilated(mask, distance=1):
    cell_ids = np.unique(mask)
    if 0 in cell_ids:
        cell_ids = cell_ids[cell_ids != 0]
    cell_ids = np.sort(cell_ids)
    n_cells = len(cell_ids)
    cell_to_index = {cell_id: idx for idx, cell_id in enumerate(cell_ids)}
    
    border_lengths = defaultdict(float)
    
    # Scan horizontal boundaries within distance
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1] - distance):
            cell1 = mask[i, j]
            cell2 = mask[i, j + distance]
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0
    
    # Scan vertical boundaries within distance
    for i in range(mask.shape[0] - distance):
        for j in range(mask.shape[1]):
            cell1 = mask[i, j]
            cell2 = mask[i + distance, j]
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0
    
    if not border_lengths:
        return coo_matrix((n_cells, n_cells)), cell_ids
    
    rows, cols, data = [], [], []
    for (idx1, idx2), length in border_lengths.items():
        rows.extend([idx1, idx2])
        cols.extend([idx2, idx1])
        data.extend([length, length])
    
    border_matrix = coo_matrix((data, (rows, cols)), shape=(n_cells, n_cells))
    return border_matrix, cell_ids

In [ ]:
"""
Minimal cellular neighborhood (CN) clustering following Schürch et al. 2020.
Input: sparse connectivity matrix + cell type labels
Output: CN clusters
"""
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from scipy import sparse

def compute_neighbor_composition(connectivity, labels):
    """
    Compute cell type composition of each cell's neighborhood.
    
    Parameters
    ----------
    connectivity : sparse matrix (n_cells, n_cells)
        Adjacency matrix (border contacts)
    labels : array-like (n_cells,)
        Cell type labels
        
    Returns
    -------
    composition : ndarray (n_cells, n_celltypes)
        Proportion of each cell type in neighborhood
    """
    labels = np.asarray(labels)
    celltypes = np.unique(labels)
    n_cells = len(labels)
    n_types = len(celltypes)
    
    # Map labels to indices
    label_to_idx = {ct: i for i, ct in enumerate(celltypes)}
    label_indices = np.array([label_to_idx[l] for l in labels])
    
    # Convert to binary matrices for each cell type
    composition = np.zeros((n_cells, n_types), dtype=np.float32)
    
    for i, ct in enumerate(celltypes):
        # Get cells of this type
        is_type = (labels == ct).astype(float)
        # Sum neighbors of this type for each cell
        neighbor_counts = connectivity.dot(is_type)
        composition[:, i] = neighbor_counts
    
    # Normalize to proportions
    totals = composition.sum(axis=1, keepdims=True)
    composition = np.divide(composition, totals, 
                           where=totals>0, 
                           out=composition)
    
    return composition, celltypes


def cluster_neighborhoods(composition, n_clusters=10, random_state=0):
    """
    Cluster cells by neighborhood composition using MiniBatchKMeans.
    
    Parameters
    ----------
    composition : ndarray (n_cells, n_celltypes)
        Output from compute_neighbor_composition
    n_clusters : int
        Number of CN clusters
    random_state : int
        Random seed
        
    Returns
    -------
    clusters : ndarray (n_cells,)
        CN cluster labels (0 to n_clusters-1)
    model : MiniBatchKMeans
        Fitted clustering model
    """
    model = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        batch_size=1000,
        n_init=10
    )
    clusters = model.fit_predict(composition)
    return clusters, model


def analyze_sample(connectivity, labels, n_clusters=10, sample_id=None):
    """
    Complete CN analysis for one sample.
    
    Parameters
    ----------
    connectivity : sparse matrix (n_cells, n_cells)
    labels : array-like (n_cells,)
        Cell type labels
    n_clusters : int
        Number of CN clusters
    sample_id : str, optional
        Sample identifier
        
    Returns
    -------
    results : dict
        'cn_labels': CN cluster assignments
        'composition': neighborhood composition matrix
        'celltypes': cell type names
        'model': clustering model
        'sample_id': sample identifier
    """
    composition, celltypes = compute_neighbor_composition(connectivity, labels)
    cn_labels, model = cluster_neighborhoods(composition, n_clusters)
    
    return {
        'cn_labels': cn_labels,
        'composition': composition,
        'celltypes': celltypes,
        'model': model,
        'sample_id': sample_id
    }


def analyze_multiple_samples(connectivity_dict, labels_dict, n_clusters=10):
    """
    Run CN analysis on multiple samples with UNIFIED clustering.
    All samples are clustered together to get consistent CN definitions.
    
    Parameters
    ----------
    connectivity_dict : dict
        {sample_id: sparse_matrix}
    labels_dict : dict
        {sample_id: labels_array}
    n_clusters : int
        Number of CN clusters (shared across all samples)
        
    Returns
    -------
    results : dict
        {sample_id: results_dict} with unified CN labels
    model : MiniBatchKMeans
        Shared clustering model
    """
    print(f"Computing neighborhood composition for {len(connectivity_dict)} samples...")
    
    # First pass: get union of all cell types
    all_celltypes = set()
    for sample_id in connectivity_dict:
        all_celltypes.update(np.unique(labels_dict[sample_id]))
    all_celltypes = sorted(all_celltypes)
    n_types = len(all_celltypes)
    celltype_to_idx = {ct: i for i, ct in enumerate(all_celltypes)}
    print(f"Found {n_types} unique cell types across all samples")
    
    # Second pass: compute aligned compositions
    compositions = []
    sample_ids = []
    cell_counts = []
    
    for sample_id in sorted(connectivity_dict.keys()):
        # Get raw composition
        comp_raw, sample_celltypes = compute_neighbor_composition(
            connectivity_dict[sample_id],
            labels_dict[sample_id]
        )
        
        # Align to unified cell type order
        comp_aligned = np.zeros((len(comp_raw), n_types), dtype=np.float32)
        for i, ct in enumerate(sample_celltypes):
            unified_idx = celltype_to_idx[ct]
            comp_aligned[:, unified_idx] = comp_raw[:, i]
        
        compositions.append(comp_aligned)
        sample_ids.append(sample_id)
        cell_counts.append(len(comp_aligned))
        print(f"  {sample_id}: {len(comp_aligned)} cells, {len(sample_celltypes)} cell types")
    
    # Stack all compositions together
    all_composition = np.vstack(compositions)
    print(f"\nClustering {all_composition.shape[0]} total cells into {n_clusters} CNs...")
    
    # Single unified clustering
    cn_labels, model = cluster_neighborhoods(all_composition, n_clusters)
    
    # Split results back by sample
    results = {}
    start_idx = 0
    for i, sample_id in enumerate(sample_ids):
        end_idx = start_idx + cell_counts[i]
        
        results[sample_id] = {
            'cn_labels': cn_labels[start_idx:end_idx],
            'composition': compositions[i],
            'celltypes': all_celltypes,  # Use unified cell types
            'model': model,
            'sample_id': sample_id
        }
        
        start_idx = end_idx
    
    return results, model


def summarize_cns(results, labels_dict):
    """
    Create summary tables of CN composition.
    
    Parameters
    ----------
    results : dict
        Output from analyze_sample or analyze_multiple_samples
    labels_dict : dict
        {sample_id: labels_array}
        
    Returns
    -------
    summary : DataFrame
        Rows = CN clusters, Cols = cell types, Values = mean proportions
    """
    summaries = []
    
    for sample_id, res in results.items():
        composition = res['composition']
        cn_labels = res['cn_labels']
        celltypes = res['celltypes']
        
        # Mean composition per CN
        df = pd.DataFrame(composition, columns=celltypes)
        df['CN'] = cn_labels
        df['sample'] = sample_id
        
        summary = df.groupby(['sample', 'CN']).mean().reset_index()
        summaries.append(summary)
    
    return pd.concat(summaries, ignore_index=True)



## Overview images

In [ ]:
channels = [
    "Ki67", "SMA", "CCR4", "CD14", "TIM3", "CD16", "CD163", "CD11b", 
    "PDL1", "CD31", "CD45", "BCMA", "CD11c", "FoxP3", "CD4", "GZMK", 
    "CD68", "CD117", "CD20", "CD8a", "CCR6", "CD56", "PD1", "CD138", 
    "GZMB", "CD127", "pSTAT1", "CD3", "CD27", "TIGIT", "CCR7", "HLADR", 
    "CD45RO", "HistonH3"
]

markers_sub = ['CD163', 'CD11b','CD31','CD45','CD4','CD68','CD20','CD8a','CD56','CD138','CD3'] + ['PD1','PDL1']

dir_ = DATA_DIR
md = pd.read_csv(dir_+'nivo_md.csv')

r=md.iloc[-5,]
    
img = tifffile.imread(f"{dir_}/compensated_imgs/{r['img']}")

for m in markers_sub:
    img_map(pctnorm(img[channels.index(m)]).T,figsize=(8,8),title=f"{r['sample_id']}\n{m}")
    plt.show()


## Import

In [ ]:
channels = [
    "Ki67", "SMA", "CCR4", "CD14", "TIM3", "CD16", "CD163", "CD11b", 
    "PDL1", "CD31", "CD45", "BCMA", "CD11c", "FoxP3", "CD4", "GZMK", 
    "CD68", "CD117", "CD20", "CD8a", "CCR6", "CD56", "PD1", "CD138", 
    "GZMB", "CD127", "pSTAT1", "CD3", "CD27", "TIGIT", "CCR7", "HLADR", 
    "CD45RO", "HistonH3"
]

dir_ = DATA_DIR
md = pd.read_csv(dir_+'nivo_md.csv')
md.head(1)

In [ ]:
pd.read_csv(dir_+'nivo_md.csv')[['group','reponse']].drop_duplicates().sort_values('group')

In [ ]:
pd.read_csv(dir_+'nivo_md.csv').sort_values('group')

In [ ]:
from tqdm import tqdm

adatas=[]

for i,r in tqdm(md.iterrows()):
    
    img = tifffile.imread(f"{dir_}/compensated_imgs/{r['img']}")
    mask = tifffile.imread(f"{dir_}/masks/{r['mask']}")
    
    img_norm = np.zeros_like(img, dtype=np.float32)
    for i in range(img.shape[0]):
        img_norm[i] = pctnorm(img[i])
    
    cellmd = summarise_cellmd(mask)
    cell_ids,expr = summarise_expression(mask, img_norm)
    adata = ad.AnnData(X=expr, obs=cellmd, var=pd.DataFrame(index=channels))
    
    for s in md.columns:
        adata.obs[s]=r[s]
    
    adatas.append(adata)

In [ ]:
adata = ad.concat(adatas)
adata

In [ ]:
#????
from inmoose.pycombat import pycombat_norm, pycombat_seq
bdata = sc.read(dir_+'outs/import.h5ad')
bdata.layers['combat'] = pycombat_norm(
    np.log1p(pd.DataFrame(bdata.X).fillna(0).values.T+1),
    bdata.obs.sample_id).T

In [ ]:
adata.write(dir_+'outs/import.h5ad',compression='gzip')

In [ ]:
dir_ = DATA_DIR
adata = sc.read(dir_+'outs/import.h5ad')

## Clustering

In [ ]:
bdata = adata.copy() #sc.pp.subsample(adata, fraction=0.1, copy=True)
bdata

In [ ]:
adata.var_names

In [ ]:
markers_sub = ['CD163', 'CD11b','CD31','CD45','CD4','CD68','CD20','CD8a','CD56','CD138','CD3']
bdata = bdata[:,markers_sub]

In [ ]:
sc.pp.pca(bdata, layer='combat')
sc.pp.neighbors(bdata,random_state=12345)
sc.tl.leiden(bdata)
sc.tl.umap(bdata)

In [ ]:
sc.tl.leiden(bdata,resolution=2,key_added='leiden_2')

In [ ]:
bdata.write(dir_+'outs/clustered.h5ad',compression='gzip')

### Phenotyping

In [ ]:
dir_ = DATA_DIR
adata = sc.read(dir_+'outs/clustered.h5ad')
adata

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(adata, color=['leiden','leiden_2'], legend_loc='on data', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(adata, color=adata.var_names, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
markers_sub = ['CD163', 'CD11b','CD31','CD45','CD4','CD68','CD20','CD8a','CD56','CD138','CD3']

sc.pl.matrixplot(adata, markers_sub, 'leiden', dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
pheno_rename = {
    '0':'Unclassified',
    '1':'Myeloid',
    '2':'Mac',
    '3':'Myeloid',
    '4':'PC',
    '5':'Mac',
    '6':'Unclassified',
    '7':'Unclassified',
    '8':'T8',
    '9':'PC',
    '10':'Myeloid',
    '11':'T4',
    '12':'Myeloid',
    '13':'PC', #PC doublet?
    '14':'MK',
    '15':'Unclassified',
    '16':'B',
    '17':'Unclassified',
    '18':'Myeloid'
}

In [ ]:
adata.obs['pheno'] = adata.obs['leiden'].map(pheno_rename)

In [ ]:
markers_sub = ['CD163', 'CD11b','CD31','CD45','CD4','CD68','CD20','CD8a','CD56','CD138','CD3']

pl = sc.pl.matrixplot(adata, markers_sub, 'pheno', dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var',
                     show=False,return_fig=True)
pl.add_totals(sort='descending', size=0.8, color='gray')
pl.show()

In [ ]:
markers_sub = ['CD163', 'CD11b','CD31','CD45','CD4','CD68','CD20','CD8a','CD56','CD138','CD3']

sc.pl.matrixplot(adata[adata.obs.pheno!='Unclassified',:], 
                 markers_sub, 'pheno', dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var', save='nivo-hm.pdf')

In [ ]:
df = adata.obs[['pt','tp','pheno']].value_counts().reset_index()\
.merge(adata.obs[['pt','tp']].value_counts().reset_index().rename({'count':'total'},axis=1))
df['pct'] = df['count']/df['total']*100
#clinical md
df=df.merge(pd.read_csv(DATA_DIR + 'nivo_clinical.csv'))
df['PC_pct'] = df['PC_pct'].astype(int)

In [ ]:
for ph in adata.obs.pheno.unique():

    pl = df[df.pheno.isin([ph])].groupby(['pt','tp','PC_pct'])['pct'].sum().reset_index()
    
    fig,ax = plt.subplots(figsize=(4,4))
    sns.scatterplot(pl, x='PC_pct', y='pct', hue='tp')
    ax.set_xlabel('PlasmaCell(%) (trephine)')
    ax.set_ylabel('% patient (IMC)')
    x = pl['PC_pct'].values
    y = pl['pct'].values
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), 
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'r--', alpha=0.5, linewidth=1, label='y=x')
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean, y_clean = x[mask], y[mask]
    z = np.polyfit(x_clean, y_clean, 1)
    p = np.poly1d(z)
    ax.plot(x_clean, p(x_clean), 'k-', linewidth=1)
    from scipy.stats import pearsonr
    r, pval = pearsonr(x_clean, y_clean)
    ax.text(0.05, 0.95, f'R²={r**2:.3f}\np={pval:.2e}', transform=ax.transAxes, va='top', fontsize=10)
    ax.set_title(ph)
    plt.show()


In [ ]:
adata

In [ ]:
# Get expression of functional markers

In [ ]:
func_mark = ['PD1','PDL1','Ki67','TIM3','GZMB','TIGIT','CD163','HLADR']
func_mark = adata.var_names.tolist()

from inmoose.pycombat import pycombat_norm, pycombat_seq
bdata = sc.read(dir_+'outs/import.h5ad')
bdata.layers['combat'] = pycombat_norm(
    np.log1p(pd.DataFrame(bdata.X).fillna(0).values.T+1),
    bdata.obs.sample_id).T

exp_raw = sc.get.obs_df(bdata,func_mark)
exp_raw.columns=[i+'_raw' for i in exp_raw.columns]
exp_norm = sc.get.obs_df(bdata,func_mark,layer='combat')
exp_norm.columns=[i+'_com' for i in exp_norm.columns]
exp = pd.concat([exp_raw,exp_norm],axis=1)

for m in exp.columns:
    adata.obs[m] = exp[m]

In [ ]:
df_ = sc.get.obs_df(bdata, bdata.var_names.tolist()+['sample_id','pt'], layer='combat')
df_['index'] = adata.obs_names
df_.to_csv(DATA_DIR + 'outs/combat_all_markers.csv')

In [ ]:
bdata.obs = bdata.obs.merge(adata.obs[['sample_id','x','y','pheno']])

In [ ]:
markers_sub = ['PD1','PDL1']

sc.pl.matrixplot(bdata, #[adata.obs.pheno!='Unclassified',:], 
                 markers_sub, 'pheno', layer='combat', dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var')

In [ ]:
clus_='pheno'
with rc_context({"figure.figsize": (5, 2.5)}):
    for m in func_mark:
        sc.pl.violin(adata,[m+'_raw'],groupby=clus_,stripplot=False,inner="box")
        sc.pl.violin(adata,[m+'_com'],groupby=clus_,stripplot=False,inner="box")

In [ ]:
for m in func_mark:

    df = adata.obs[['pheno',m+'_com']]
    
    df['decile'] = [i+1 for i in pd.qcut(df[m+'_com'], q=20, labels=False, duplicates='drop')]
    decile_counts = df.groupby(['decile', 'pheno']).size().unstack(fill_value=0)
    decile_pcts = decile_counts.div(decile_counts.sum(axis=1), axis=0) * 100
    
    decile_pcts.plot(kind='bar', stacked=True, width=0.9, figsize=(5,3))
    plt.xlabel('ntile')
    plt.ylabel('%')
    plt.legend(title='Phenotype', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(m)
    plt.tight_layout()
    plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(4,4))
sc.pl.scatter(adata, x='CD11b', y='Ki67_com', color='pheno', layers='combat', ax=ax)

In [ ]:
fig,ax = plt.subplots(figsize=(4,4))
sc.pl.scatter(adata, x='CD8a', y='TIGIT_com', color='pheno', layers='combat', ax=ax)

In [ ]:
###adata = sc.read(DATA_DIR + 'outs/phenotyped.h5ad')

In [ ]:
adata.write(DATA_DIR + 'outs/phenotyped.h5ad',compression='gzip')

In [ ]:
adata.obs.to_csv(DATA_DIR + 'outs/phenotyped-obs.csv',index=False)

## Clustering 2: T cell/Myeloid subsets

In [ ]:
dir_ = DATA_DIR
adata_raw = sc.read(dir_+'outs/import.h5ad')
adata = sc.read(DATA_DIR + 'outs/phenotyped.h5ad')

In [ ]:
adata

### CD8+ subsets

In [ ]:
bdata = adata_raw[adata_raw.obs_names.isin(adata[adata.obs.pheno=='T8'].obs_names),:]

In [ ]:
markers_sub = ['Ki67', 'TIM3','GZMK','PD1','GZMB','CD127','CD27','TIGIT','CCR7','HLADR','CD45RO']
bdata = bdata[:,markers_sub]

In [ ]:
from inmoose.pycombat import pycombat_norm, pycombat_seq
bdata.layers['combat'] = pycombat_norm(
    np.log1p(pd.DataFrame(bdata.X).fillna(0).values.T+1),
    bdata.obs.sample_id).T

sc.pp.pca(bdata, layer='combat')
sc.pp.neighbors(bdata,random_state=12345)
sc.tl.leiden(bdata)
sc.tl.leiden(bdata,resolution=2,key_added='leiden_2')
sc.tl.umap(bdata)

In [ ]:
sc.tl.leiden(bdata,resolution=0.5,key_added='leiden_0.5')

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=['leiden_0.5','leiden'], legend_loc='on data', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=bdata.var_names, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
markers_sub = bdata.var_names

sc.pl.matrixplot(bdata, markers_sub, 'leiden_0.5', dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
pheno_rename = {
    '0':'Other',
    '1':'Tn',
    '2':'Active',
    '3':'Tem',
    '4':'Prolif',
    '5':'Tn',
    '6':'Tm',
    '7':'Tm',
    '8':'TIM3+',
    '9':'Tem'
}

In [ ]:
bdata.obs['pheno'] = bdata.obs['leiden_0.5'].map(pheno_rename)

In [ ]:
dir_ = DATA_DIR
bdata.write(dir_+'outs/phenotyped_T8.h5ad',compression='gzip')
bdata.obs.to_csv(dir_+'outs/phenotyped_T8-obs.csv',index=False)

### CD4+ subsets

In [ ]:
bdata = adata_raw[adata_raw.obs_names.isin(adata[adata.obs.pheno=='T4'].obs_names),:]

In [ ]:
bdata.var_names

In [ ]:
markers_sub = ['Ki67', 'TIM3','GZMK','PD1','GZMB','CD127','CD27','TIGIT','CCR7','HLADR','CD45RO','FoxP3']
bdata = bdata[:,markers_sub]

In [ ]:
from inmoose.pycombat import pycombat_norm, pycombat_seq
bdata.layers['combat'] = pycombat_norm(
    np.log1p(pd.DataFrame(bdata.X).fillna(0).values.T+1),
    bdata.obs.sample_id).T

sc.pp.pca(bdata, layer='combat')
sc.pp.neighbors(bdata,random_state=12345)
sc.tl.leiden(bdata)
sc.tl.leiden(bdata,resolution=2,key_added='leiden_2')
sc.tl.umap(bdata)

In [ ]:
sc.tl.leiden(bdata,resolution=0.5,key_added='leiden_0.5')

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=['leiden_0.5','leiden'], legend_loc='on data', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=bdata.var_names, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
markers_sub = bdata.var_names

sc.pl.matrixplot(bdata, markers_sub, 'leiden_0.5', dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
pheno_rename = {
    '0':'Active',
    '1':'Other',
    '2':'GZMB',
    '3':'RO+',
    '4':'Tn',
    '5':'Treg_cycle',
    '6':'Cycle',
    '7':'PanPos',
    '8':'TIM3+',
}

In [ ]:
bdata.obs['pheno'] = bdata.obs['leiden_0.5'].map(pheno_rename)

In [ ]:
dir_ = DATA_DIR
bdata.write(dir_+'outs/phenotyped_T4.h5ad',compression='gzip')
bdata.obs.to_csv(dir_+'outs/phenotyped_T4-obs.csv',index=False)

### Myeloid subsets

In [ ]:
bdata = adata_raw[adata_raw.obs_names.isin(adata[adata.obs.pheno=='Myeloid'].obs_names),:]

In [ ]:
bdata.var_names

In [ ]:
markers_sub = ['CD14','CD16','CD11b', 'PDL1','CD11c','CD45RO']
bdata = bdata[:,markers_sub]

In [ ]:
from inmoose.pycombat import pycombat_norm, pycombat_seq
bdata.layers['combat'] = pycombat_norm(
    np.log1p(pd.DataFrame(bdata.X).fillna(0).values.T+1),
    bdata.obs.sample_id).T

sc.pp.pca(bdata, layer='combat')
sc.pp.neighbors(bdata,random_state=12345)
sc.tl.leiden(bdata)
sc.tl.leiden(bdata,resolution=2,key_added='leiden_2')
sc.tl.umap(bdata)

In [ ]:
sc.tl.leiden(bdata,resolution=0.5,key_added='leiden_0.5')

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=['leiden_0.5','leiden'], legend_loc='on data', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=bdata.var_names, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
markers_sub = bdata.var_names

sc.pl.matrixplot(bdata, markers_sub, 'leiden_0.5', dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
pheno_rename = {
    '0':'Active',
    '1':'Other',
    '2':'GZMB',
    '3':'RO+',
    '4':'Tn',
    '5':'Treg_cycle',
    '6':'Cycle',
    '7':'PanPos',
    '8':'TIM3+',
}

In [ ]:
bdata.obs['pheno'] = bdata.obs['leiden_0.5'].map(pheno_rename)

In [ ]:
dir_ = DATA_DIR
bdata.write(dir_+'outs/phenotyped_Mye.h5ad',compression='gzip')
bdata.obs.to_csv(dir_+'outs/phenotyped_Mye-obs.csv',index=False)

### CD4+ subsets

In [ ]:
bdata = adata_raw[adata_raw.obs_names.isin(adata[adata.obs.pheno=='T4'].obs_names),:]

In [ ]:
bdata.var_names

In [ ]:
markers_sub = ['Ki67', 'TIM3','GZMK','PD1','GZMB','CD127','CD27','TIGIT','CCR7','HLADR','CD45RO','FoxP3']
bdata = bdata[:,markers_sub]

In [ ]:
from inmoose.pycombat import pycombat_norm, pycombat_seq
bdata.layers['combat'] = pycombat_norm(
    np.log1p(pd.DataFrame(bdata.X).fillna(0).values.T+1),
    bdata.obs.sample_id).T

sc.pp.pca(bdata, layer='combat')
sc.pp.neighbors(bdata,random_state=12345)
sc.tl.leiden(bdata)
sc.tl.leiden(bdata,resolution=2,key_added='leiden_2')
sc.tl.umap(bdata)

In [ ]:
sc.tl.leiden(bdata,resolution=0.5,key_added='leiden_0.5')

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=['leiden_0.5','leiden'], legend_loc='on data', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(bdata, color=bdata.var_names, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
markers_sub = bdata.var_names

sc.pl.matrixplot(bdata, markers_sub, 'leiden_0.5', dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
pheno_rename = {
    '0':'Active',
    '1':'Other',
    '2':'GZMB',
    '3':'RO+',
    '4':'Tn',
    '5':'Treg_cycle',
    '6':'Cycle',
    '7':'PanPos',
    '8':'TIM3+',
}

In [ ]:
bdata.obs['pheno'] = bdata.obs['leiden_0.5'].map(pheno_rename)

In [ ]:
dir_ = DATA_DIR
bdata.write(dir_+'outs/phenotyped_T4.h5ad',compression='gzip')
bdata.obs.to_csv(dir_+'outs/phenotyped_T4-obs.csv',index=False)

## Spatial Analysis

### Basic distances (failed)

In [ ]:
obs = pd.read_csv(DATA_DIR + 'outs/phenotyped-obs.csv',)
obs.head(1)

In [ ]:
#1414px=2000um
#1414/10px = 141px = 2000/10 = 200um
#1414/100px = 14.14 px = 2000/100 = 10um

D=14.14

In [ ]:
import pandas as pd
from scipy.spatial.distance import cdist

def label_proximity_df(df, label_col, x_col='x', y_col='y', distance=D):
    coords = df[[x_col, y_col]].values
    labels = df[label_col].values
    unique_labels = df[label_col].unique()
    
    dist_matrix = cdist(coords, coords)
    
    results = []
    for i, label_a in enumerate(unique_labels):
        mask_a = labels == label_a
        for j, label_b in enumerate(unique_labels[i:], i):
            mask_b = labels == label_b
            count = ((dist_matrix[mask_a][:, mask_b] <= distance) & 
                    (dist_matrix[mask_a][:, mask_b] > 0)).sum()
            
            total = (labels == label_a).sum() * (labels == label_b).sum()
            results.append({
                'from': label_a,
                'to': label_b,
                'count': count,
                'norm': count / total
            })
    
    return pd.DataFrame(results)

In [ ]:
results = []

for sample_id in obs.sample_id.unique():

    obs_sub = obs[obs.sample_id == sample_id]
    results_sub = label_proximity_df(obs_sub, 'pheno')
    results_sub['sample_id'] = sample_id
    results.append(results_sub)

In [ ]:
#pd.concat(results,axis=0).to_csv(DATA_DIR + 'outs/distances-D50.csv',index=False)
pd.concat(results,axis=0).to_csv(DATA_DIR + 'outs/distances-D14.csv',index=False)

### Re-do CN

In [ ]:
obs_data = pd.read_csv(DATA_DIR + 'outs/phenotyped-obs.csv',)
obs_data.head(1)

In [ ]:
# Calculate borders

# DILATED border, 10um as per original
dist_10um_pixels = int(1440/2000*10)

sample_ids = obs_data.sample_id.unique()

for sample_id in sample_ids:

    #import
    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample.index = [i+1 for i in range(obs_sample.shape[0])]
    mask = tifffile.imread(fDATA_DIR + 'masks/{ obs_sample['mask'].tolist()[0] }")
    
    #overlapping cells, index
    shared_cells = np.intersect1d(mask,obs_sample.index)
    mask = np.where(np.isin(mask,shared_cells),mask,0)

    # Calc cell border length - dilated distance
    border_lengths,cell_ids = calculate_cell_border_lengths_dilated(mask, dist_10um_pixels )
    
    #save results
    from scipy.sparse import save_npz
    save_npz(
        fDATA_DIR + 'borders/{sample_id}-border_lengths_dilated.npz', 
        border_lengths) 
    pd.DataFrame({"Cell_IDs":cell_ids}).to_csv(
        fDATA_DIR + 'borders/{sample_id}-border_lengths_cellids.npz',index=False
    )


In [ ]:
# Import data, assemble

from scipy.sparse import load_npz

sample_ids = obs_data.sample_id.unique()

#Rename subsets
obs_data['Ki67_pos'] = obs_data['Ki67_com'].rank(pct=True) > 0.95
obs_data['PDL1_pos'] = obs_data['PDL1_com'].rank(pct=True) > 0.95
obs_data['HLADR_pos'] = obs_data['HLADR_com'].rank(pct=True) > 0.95
obs_data['pheno_func'] = obs_data['pheno'].tolist()
obs_data['pheno_func'] = np.where((obs_data['pheno'] == 'PC') & (obs_data['Ki67_pos'] == True),'PC_Ki67',obs_data['pheno_func'])
obs_data['pheno_func'] = np.where((obs_data['pheno'] == 'PC') & (obs_data['PDL1_pos'] == True),'PC_PDL1',obs_data['pheno_func'])
obs_data['pheno_func'] = np.where((obs_data['pheno'] == 'Myeloid') & (obs_data['PDL1_pos'] == True),'Myeloid_PDL1',obs_data['pheno_func'])
obs_data['pheno_func'] = np.where((obs_data['pheno'] == 'T4') & (obs_data['HLADR_pos'] == True),'T4_HLADR',obs_data['pheno_func'])
obs_data['pheno_func'] = np.where((obs_data['pheno'] == 'T8') & (obs_data['HLADR_pos'] == True),'T8_HLADR',obs_data['pheno_func'])

clus = 'pheno_func'
clus_remove = ['Unclassified']

connectivity_dict = {}
labels_dict = {}
obs_dict = {}

for sample_id in sample_ids:

    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample.index = [i+1 for i in range(obs_sample.shape[0])]
    
    border_lengths = load_npz(fDATA_DIR + 'borders/{sample_id}-border_lengths_dilated.npz')
    cell_ids = pd.read_csv(fDATA_DIR + 'borders/{sample_id}-border_lengths_cellids.npz')['Cell_IDs'].tolist()

    #subset
    obs_sample = obs_sample[~obs_sample[clus].isin(clus_remove)]
    
    #intersection of cell IDs
    cells_use = np.intersect1d(obs_sample.index, cell_ids)
    #integer indices for matrix slicing
    idx = np.searchsorted(cell_ids, cells_use)
    #slice matrix and obs
    border_lengths = border_lengths.tocsr()[idx, :][:, idx].tocoo()
    obs_sample = obs_sample.loc[cells_use]
    
    labels = obs_sample[clus].tolist()
    
    connectivity_dict[sample_id] = border_lengths
    labels_dict[sample_id] = labels
    obs_dict[sample_id] = obs_sample.copy()

In [ ]:
# Run over CNs

n_cluster_options = [5,8,10]
saveloc = DATA_DIR + 'CN_data/'
run_ = 'pheno_func'

for n_clusters in n_cluster_options:
    
    results = analyze_multiple_samples(connectivity_dict=connectivity_dict,labels_dict=labels_dict,n_clusters=n_clusters)
    
    for sample_id in results[0].keys():
        obs_dict[sample_id]['cn_labels'] = ['CN'+str(i) for i in results[0][sample_id]['cn_labels'].tolist()]
    
    df = pd.concat(obs_dict.values(),axis=0).reset_index()
    df.to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-cells_data.csv",index=False)

    df_in=df
    clus="cn_labels"
    
    clus_ids = df_in[clus].unique().tolist()
    
    df_in = df_in[['sample_id',clus]].value_counts().reset_index()\
        .merge(df_in[['sample_id']].value_counts().reset_index().rename({'count':'total'},axis=1))
       
    df_in['pct'] = df_in['count']/df_in['total']
    df_in = df_in.pivot(index=['sample_id'],columns=clus, values='pct').fillna(0).reset_index()
    df_in[clus_ids] = df_in[clus_ids]*100
    df_in.to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-sample_pct.csv",index=False)

## Publication plots


In [ ]:
adata = sc.read(DATA_DIR + 'outs/phenotyped.h5ad')
adata

In [ ]:
adata.obs.pheno.value_counts()

In [ ]:
sc.settings.figdir = DATA_DIR + 'figures/'

adata.obs['PD-L1'] = adata.obs['PDL1_com']
markers_sub = ['CD163', 'CD11b','CD31','CD45','CD4','CD68','CD20','CD8a','CD56','CD138','CD3','PD-L1']

sc.pl.matrixplot(adata[adata.obs.pheno!='Unclassified',:], 
                 markers_sub, 'pheno', dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var', save='markers.pdf')

sc.pl.matrixplot(adata[adata.obs.pheno!='Unclassified',:], 
                 markers_sub, 'pheno', dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var', save='markers.svg')

## Maps

In [ ]:
from imclib.imctoolkit import *

In [ ]:
# CN3 (PC)

s_id='Patient1_Baseline_001'
CN_focus='CN3'

mask = tifffile.imread(fDATA_DIR + 'masks/{s_id}.tiff')
gdf = mask2shapely(mask)
gdf['x_int'] = gdf['x'].astype(int)
gdf['y_int'] = gdf['y'].astype(int)

obs = pd.read_csv(DATA_DIR + 'CN_data/pheno_8clus-cells_data.csv')
obs = obs[obs.sample_id==s_id]
obs['y_int'] = obs['y'].astype(int)
obs['x_int'] = obs['x'].astype(int)

gdf = gdf.merge(obs[['x_int', 'y_int', 'pheno', 'cn_labels']], left_on=['x_int', 'y_int'], right_on=['x_int', 'y_int'], how='left')

gdf_sub = gdf[(gdf['x'] < 250) & (gdf['y'] > 350)]

gdf_sub['pheno'].fillna('Unclassified', inplace=True)
gdf_sub['color'] = gdf_sub['pheno'].map({
    'MK': 'gold',
    'Myeloid': '#C7E9B4',
    'Unclassified': 'lightgrey',
    'T4': '#6A51A3',
    'Mac': '#238B45',
    'PC': '#FF0000',
    'T8': '#08519C',
    'B': '#C49A6C'
})

fig, ax = plt.subplots(figsize=(5,5))
gdf_sub.plot(aspect=1, ax=ax, edgecolor='black', linewidth=0.75, alpha=1, color=gdf_sub["color"])
import matplotlib.patches as mpatches
unique_entries = gdf_sub.drop_duplicates(subset=["pheno", "color"])
handles = [mpatches.Patch(facecolor=row["color"], edgecolor='black', label=row["pheno"])
    for _, row in unique_entries.iterrows()]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), ncol=1, loc="upper left")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_axis_off()

# Scalebar: 2000 px = 1414 um
_um_per_px = 1414 / 2000
_sb_um = 20
_sb_len = _sb_um / _um_per_px
_xl, _yl = ax.get_xlim(), ax.get_ylim()
_x0 = _xl[0] + 0.01 * (_xl[1] - _xl[0])
_y0 = _yl[0] + 0.01 * (_yl[1] - _yl[0])
ax.plot([_x0, _x0 + _sb_len], [_y0, _y0], color='black', lw=2, solid_capstyle='butt')
ax.text(_x0 + _sb_len / 2, _y0 + 0.01 * (_yl[1] - _yl[0]), f'{_sb_um} μm', ha='center', va='bottom', fontsize=8)

plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_pheno.pdf', dpi=300, bbox_inches='tight', transparent=False)
plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_pheno.svg', dpi=300, bbox_inches='tight', transparent=False)
plt.show()

fig, ax = plt.subplots(figsize=(5,5))
gdf_sub['color'] = ['black' if label == CN_focus else 'lightgrey' for label in gdf_sub['cn_labels']]
gdf_sub.plot(aspect=1, ax=ax, edgecolor='black', linewidth=0.75, alpha=1, color=gdf_sub["color"])
unique_entries = gdf_sub.drop_duplicates(subset=["cn_labels", "color"])
handles = [mpatches.Patch(facecolor=row["color"], edgecolor='black', label=row["cn_labels"])
    for _, row in unique_entries.iterrows()]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), ncol=1, loc="upper left")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_axis_off()

# Scalebar
_xl, _yl = ax.get_xlim(), ax.get_ylim()
_x0 = _xl[0] + 0.01 * (_xl[1] - _xl[0])
_y0 = _yl[0] + 0.01 * (_yl[1] - _yl[0])
ax.plot([_x0, _x0 + _sb_len], [_y0, _y0], color='black', lw=2, solid_capstyle='butt')
ax.text(_x0 + _sb_len / 2, _y0 + 0.01 * (_yl[1] - _yl[0]), f'{_sb_um} μm', ha='center', va='bottom', fontsize=8)

plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_CN.pdf', dpi=300, bbox_inches='tight', transparent=False)
plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_CN.svg', dpi=300, bbox_inches='tight', transparent=False)
plt.show()
plt.show()

In [ ]:
# CN3 (PC)

s_id='Patient6_PostNivo_004'
CN_focus='CN2'

mask = tifffile.imread(fDATA_DIR + 'masks/{s_id}.tiff')
gdf = mask2shapely(mask)
gdf['x_int'] = gdf['x'].astype(int)
gdf['y_int'] = gdf['y'].astype(int)

obs = pd.read_csv(DATA_DIR + 'CN_data/pheno_8clus-cells_data.csv')
obs = obs[obs.sample_id==s_id]
obs['y_int'] = obs['y'].astype(int)
obs['x_int'] = obs['x'].astype(int)

gdf = gdf.merge(obs[['x_int', 'y_int', 'pheno', 'cn_labels']], left_on=['x_int', 'y_int'], right_on=['x_int', 'y_int'], how='left')

gdf_sub = gdf[(gdf['x'] < 300) & (gdf['y'] > 200) & (gdf['y'] < 500)]

gdf_sub['pheno'].fillna('Unclassified', inplace=True)
gdf_sub['color'] = gdf_sub['pheno'].map({
    'MK': 'gold',
    'Myeloid': '#C7E9B4',
    'Unclassified': 'lightgrey',
    'T4': '#6A51A3',
    'Mac': '#238B45',
    'PC': '#FF0000',
    'T8': '#08519C',
    'B': '#C49A6C'
})


fig, ax = plt.subplots(figsize=(5,5))
gdf_sub.plot(aspect=1, ax=ax, edgecolor='black', linewidth=0.75, alpha=1, color=gdf_sub["color"])
import matplotlib.patches as mpatches
unique_entries = gdf_sub.drop_duplicates(subset=["pheno", "color"])
handles = [mpatches.Patch(facecolor=row["color"], edgecolor='black', label=row["pheno"])
    for _, row in unique_entries.iterrows()]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), ncol=1, loc="upper left")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_axis_off()

# Scalebar: 2000 px = 1414 um
_um_per_px = 1414 / 2000
_sb_um = 20
_sb_len = _sb_um / _um_per_px
_xl, _yl = ax.get_xlim(), ax.get_ylim()
_x0 = _xl[0] + 0.01 * (_xl[1] - _xl[0])
_y0 = _yl[0] + 0.01 * (_yl[1] - _yl[0])
ax.plot([_x0, _x0 + _sb_len], [_y0, _y0], color='black', lw=2, solid_capstyle='butt')
ax.text(_x0 + _sb_len / 2, _y0 + 0.01 * (_yl[1] - _yl[0]), f'{_sb_um} μm', ha='center', va='bottom', fontsize=8)

plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_pheno.pdf', dpi=300, bbox_inches='tight', transparent=False)
plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_pheno.svg', dpi=300, bbox_inches='tight', transparent=False)
plt.show()

fig, ax = plt.subplots(figsize=(5,5))
gdf_sub['color'] = ['black' if label == CN_focus else 'lightgrey' for label in gdf_sub['cn_labels']]
gdf_sub.plot(aspect=1, ax=ax, edgecolor='black', linewidth=0.75, alpha=1, color=gdf_sub["color"])
unique_entries = gdf_sub.drop_duplicates(subset=["cn_labels", "color"])
handles = [mpatches.Patch(facecolor=row["color"], edgecolor='black', label=row["cn_labels"])
    for _, row in unique_entries.iterrows()]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), ncol=1, loc="upper left")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_axis_off()

# Scalebar
_xl, _yl = ax.get_xlim(), ax.get_ylim()
_x0 = _xl[0] + 0.01 * (_xl[1] - _xl[0])
_y0 = _yl[0] + 0.01 * (_yl[1] - _yl[0])
ax.plot([_x0, _x0 + _sb_len], [_y0, _y0], color='black', lw=2, solid_capstyle='butt')
ax.text(_x0 + _sb_len / 2, _y0 + 0.01 * (_yl[1] - _yl[0]), f'{_sb_um} μm', ha='center', va='bottom', fontsize=8)

plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_CN.pdf', dpi=300, bbox_inches='tight', transparent=False)
plt.savefig(fDATA_DIR + 'figures/map-{CN_focus}_CN.svg', dpi=300, bbox_inches='tight', transparent=False)
plt.show()

In [ ]:
# One large slide


s_id='Patient8_Baseline_001'

mask = tifffile.imread(fDATA_DIR + 'masks/{s_id}.tiff')

gdf = mask2shapely(mask)
gdf['x_int'] = gdf['x'].astype(int)
gdf['y_int'] = gdf['y'].astype(int)

obs = pd.read_csv(DATA_DIR + 'CN_data/pheno_8clus-cells_data.csv')
obs = obs[obs.sample_id==s_id]
obs['y_int'] = obs['y'].astype(int)
obs['x_int'] = obs['x'].astype(int)

gdf = gdf.merge(obs[['x_int', 'y_int', 'pheno', 'cn_labels']], left_on=['x_int', 'y_int'], right_on=['x_int', 'y_int'], how='left')

gdf['pheno'].fillna('Unclassified', inplace=True)
gdf['color'] = gdf['pheno'].map({
    'MK': 'gold',
    'Myeloid': '#C7E9B4',
    'Unclassified': 'lightgrey',
    'T4': '#6A51A3',
    'Mac': '#238B45',
    'PC': '#FF0000',
    'T8': '#08519C',
    'B': '#C49A6C'
})

fig, ax = plt.subplots(figsize=(12,12))
gdf.plot(aspect=1, ax=ax, edgecolor='black', linewidth=0.75, alpha=1, color=gdf["color"])
import matplotlib.patches as mpatches
unique_entries = gdf.drop_duplicates(subset=["pheno", "color"])
handles = [mpatches.Patch(facecolor=row["color"], edgecolor='black', label=row["pheno"])
    for _, row in unique_entries.iterrows()]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), ncol=1, loc="upper left")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_axis_off()

# Scalebar: 2000 px = 1414 um
_um_per_px = 1414 / 2000
_sb_um = 20
_sb_len = _sb_um / _um_per_px
_xl, _yl = ax.get_xlim(), ax.get_ylim()
_x0 = _xl[0] + 0.01 * (_xl[1] - _xl[0])
_y0 = _yl[0] + 0.01 * (_yl[1] - _yl[0])
ax.plot([_x0, _x0 + _sb_len], [_y0, _y0], color='black', lw=2, solid_capstyle='butt')
ax.text(_x0 + _sb_len / 2, _y0 + 0.01 * (_yl[1] - _yl[0]), f'{_sb_um} μm', ha='center', va='bottom', fontsize=8)

plt.savefig(fDATA_DIR + 'figures/map-large_pheno.pdf', dpi=300, bbox_inches='tight', transparent=False)
plt.savefig(fDATA_DIR + 'figures/map-large_pheno.svg', dpi=300, bbox_inches='tight', transparent=False)
plt.show()


In [ ]:
# One large slide


s_id='Patient1_PostNivo_001'

mask = tifffile.imread(fDATA_DIR + 'masks/{s_id}.tiff')

gdf = mask2shapely(mask)
gdf['x_int'] = gdf['x'].astype(int)
gdf['y_int'] = gdf['y'].astype(int)

obs = pd.read_csv(DATA_DIR + 'CN_data/pheno_8clus-cells_data.csv')
obs = obs[obs.sample_id==s_id]
obs['y_int'] = obs['y'].astype(int)
obs['x_int'] = obs['x'].astype(int)

gdf = gdf.merge(obs[['x_int', 'y_int', 'pheno', 'cn_labels']], left_on=['x_int', 'y_int'], right_on=['x_int', 'y_int'], how='left')

gdf['pheno'].fillna('Unclassified', inplace=True)
gdf['color'] = gdf['pheno'].map({
    'MK': 'gold',
    'Myeloid': '#C7E9B4',
    'Unclassified': 'lightgrey',
    'T4': '#6A51A3',
    'Mac': '#238B45',
    'PC': '#FF0000',
    'T8': '#08519C',
    'B': '#C49A6C'
})

fig, ax = plt.subplots(figsize=(12,12))
gdf.plot(aspect=1, ax=ax, edgecolor='black', linewidth=0.75, alpha=1, color=gdf["color"])
import matplotlib.patches as mpatches
unique_entries = gdf.drop_duplicates(subset=["pheno", "color"])
handles = [mpatches.Patch(facecolor=row["color"], edgecolor='black', label=row["pheno"])
    for _, row in unique_entries.iterrows()]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), ncol=1, loc="upper left")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_axis_off()

# Scalebar: 2000 px = 1414 um
_um_per_px = 1414 / 2000
_sb_um = 20
_sb_len = _sb_um / _um_per_px
_xl, _yl = ax.get_xlim(), ax.get_ylim()
_x0 = _xl[0] + 0.01 * (_xl[1] - _xl[0])
_y0 = _yl[0] + 0.01 * (_yl[1] - _yl[0])
ax.plot([_x0, _x0 + _sb_len], [_y0, _y0], color='black', lw=2, solid_capstyle='butt')
ax.text(_x0 + _sb_len / 2, _y0 + 0.01 * (_yl[1] - _yl[0]), f'{_sb_um} μm', ha='center', va='bottom', fontsize=8)

plt.savefig(fDATA_DIR + 'figures/map-large_pheno2.pdf', dpi=300, bbox_inches='tight', transparent=False)
plt.savefig(fDATA_DIR + 'figures/map-large_pheno2.svg', dpi=300, bbox_inches='tight', transparent=False)
plt.show()
